Generate a chart from a preprocessed `.pt` song

Runs top to bottom: setup, load model, define the generator, then set song settings and run.

In [1]:
from pathlib import Path

from tracers.project import setup_project
from tracers.pipelines.trainer_factory import build_trainer

PROJECT_ROOT = Path.cwd().parent
setup = setup_project(PROJECT_ROOT)

MODEL_KEY = "tracer_np"

experiment_config = setup["experiment_config"]
hyperparams = setup["hyperparams"]
segment_frames = setup["segment_frames"]
device = setup["device"]

In [2]:
from tracers.pipelines.generate import run_custom_generation

trainer, model = build_trainer(
    model_key=MODEL_KEY,
    hyperparams=hyperparams,
    experiment_config=experiment_config,
    device=device,
    checkpoint_path=str(PROJECT_ROOT / "notebooks" / "checkpoints" / "tracer_np" / "best.pt"),
    train_loader=None,
    valid_loader=None,
)

Loaded (epoch 76, best val loss 0.1494)


In [3]:
from tracers.export.chart import export_generated_chart

# ---------- settings ----------
AUDIO_PT_NAME = "renai"
AUDIO_PT = PROJECT_ROOT / f"{AUDIO_PT_NAME}.pt"
DIFFICULTY = 5
BPM = 120
OFFSET_MS = 0
LANE_MAP = (0, 2, 4, 6)

CONTEXT_FRAMES = 0
TEMPERATURE = 0.6
TOP_K = None
TOP_P = None
SEED = 0
SAVE_DEBUG_CSV = True

# ---------- generate ----------
generated, debug_segments = run_custom_generation(
    trainer,
    AUDIO_PT,
    difficulty=DIFFICULTY,
    segment_frames=segment_frames,
    context_frames=CONTEXT_FRAMES,
    seed_action=0,
    seed=SEED,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    collect_debug=SAVE_DEBUG_CSV,
)

# ---------- export ----------
output_dir = Path(f"charts/{AUDIO_PT_NAME}")
output_dir.mkdir(parents=True, exist_ok=True)

# Shared metadata tag string based on your settings
run_meta = (
    f"_diff{DIFFICULTY}"
    f"_ctx{CONTEXT_FRAMES}"
    f"_temp{TEMPERATURE}"
    f"_topk{TOP_K if TOP_K is not None else 'none'}"
    f"_topp{TOP_P if TOP_P is not None else 'none'}"
    f"_seed{SEED}"
)

chart_filename = f"chart{run_meta}.json"
debug_filename = f"debug{run_meta}.csv"

chart_path = output_dir / chart_filename
debug_csv_path = output_dir / debug_filename

# Export chart json with metadata in filename
export_generated_chart(
    generated,
    chart_path,
    bpm=BPM,
    start_frame=0,
    offset_ms=OFFSET_MS,
    lane_map=LANE_MAP,
)
print(f"Saved {chart_filename}")

# Export diagnostic CSV with identical metadata
if SAVE_DEBUG_CSV:
    from tracers.generate.converter import export_generation_debug
    export_generation_debug(
        debug_segments, 
        output_path=str(debug_csv_path), 
        top_k=5
    )
    print(f"Saved {debug_filename}")

Prediction frames: 25475 (254.8 s)
Window size: 2000 | Context: 0 | Stride: 2000
Window      1-  2000 | Context:    0 | New: 2000 | Active:   38
Window   2001-  4000 | Context:    0 | New: 2000 | Active:   93
Window   4001-  6000 | Context:    0 | New: 2000 | Active:   84
Window   6001-  8000 | Context:    0 | New: 2000 | Active:  123
Window   8001- 10000 | Context:    0 | New: 2000 | Active:  121
Window  10001- 12000 | Context:    0 | New: 2000 | Active:   99
Window  12001- 14000 | Context:    0 | New: 2000 | Active:   76
Window  14001- 16000 | Context:    0 | New: 2000 | Active:   88
Window  16001- 18000 | Context:    0 | New: 2000 | Active:   80
Window  18001- 20000 | Context:    0 | New: 2000 | Active:   95
Window  20001- 22000 | Context:    0 | New: 2000 | Active:   78
Window  22001- 24000 | Context:    0 | New: 2000 | Active:  105
Window  24001- 25475 | Context:    0 | New: 1475 | Active:   20

Done. Generated 25475 frames. Active frames: 1100 / 25475
Saved chart_diff5_ctx0_temp0

C:\Users\joshua\vsjycode\csc173\transformers\TRACERS\src\tracers\generate\converter.py:72: UserWarning: Unclosed long note in lane 4 before frame 23317
  warnings.warn(
C:\Users\joshua\vsjycode\csc173\transformers\TRACERS\src\tracers\generate\converter.py:72: UserWarning: Unclosed long note in lane 2 before frame 23343
  warnings.warn(


Exported 25475 frame records to: charts\renai\debug_diff5_ctx0_temp0.6_topknone_toppnone_seed0.csv
Saved debug_diff5_ctx0_temp0.6_topknone_toppnone_seed0.csv


Experiments

In [4]:
from tracers.pipelines.generate import run_generation_sweep

In [5]:
context_sweep = [
    {
        "name": "greedy_ctx0",
        "context_frames": 0,
        "temperature": None,
    },
    {
        "name": "greedy_ctx250",
        "context_frames": 250,
        "temperature": None,
    },
    {
        "name": "greedy_ctx500",
        "context_frames": 500,
        "temperature": None,
    },
    {
        "name": "greedy_ctx1000",
        "context_frames": 1000,
        "temperature": None,
    },
]


results_context = run_generation_sweep(
    trainer,
    AUDIO_PT,
    difficulty=DIFFICULTY,
    segment_frames=segment_frames,
    configs=context_sweep,
    bpm=BPM,
    offset_ms=OFFSET_MS,
    lane_map=LANE_MAP,
    default_seed=SEED,
    save_debug_csv=True,
)

Generating sweep for: renai
Number of configurations: 4
Output directory: charts\renai

RUN 1/4: greedy_ctx0
Context: 0 | Temperature: None | Top-K: None | Top-P: None | Seed: 0
Prediction frames: 25475 (254.8 s)
Window size: 2000 | Context: 0 | Stride: 2000
Window      1-  2000 | Context:    0 | New: 2000 | Active:    0
Window   2001-  4000 | Context:    0 | New: 2000 | Active:    0
Window   4001-  6000 | Context:    0 | New: 2000 | Active:    0
Window   6001-  8000 | Context:    0 | New: 2000 | Active:    0
Window   8001- 10000 | Context:    0 | New: 2000 | Active:    0
Window  10001- 12000 | Context:    0 | New: 2000 | Active:    0
Window  12001- 14000 | Context:    0 | New: 2000 | Active:    0
Window  14001- 16000 | Context:    0 | New: 2000 | Active:   52
Window  16001- 18000 | Context:    0 | New: 2000 | Active:    0
Window  18001- 20000 | Context:    0 | New: 2000 | Active:    1
Window  20001- 22000 | Context:    0 | New: 2000 | Active:    0
Window  22001- 24000 | Context:    0 

In [7]:
temperature_sweep = [
    {
        "name": "temp01",
        "context_frames": 250,
        "temperature": 0.1,
    },
    {
        "name": "temp02",
        "context_frames": 250,
        "temperature": 0.2,
    },
    {
        "name": "temp03",
        "context_frames": 250,
        "temperature": 0.3,
    },

    {
        "name": "temp04",
        "context_frames": 250,
        "temperature": 0.1,
    },
    {
        "name": "temp05",
        "context_frames": 250,
        "temperature": 0.5,
    },
    {
        "name": "temp06",
        "context_frames": 250,
        "temperature": 0.6,
    }
]

results_temperature = run_generation_sweep(
    trainer,
    AUDIO_PT,
    difficulty=DIFFICULTY,
    segment_frames=segment_frames,
    configs=temperature_sweep,
    bpm=BPM,
    offset_ms=OFFSET_MS,
    lane_map=LANE_MAP,
    default_seed=SEED,
    save_debug_csv=True,
)

Generating sweep for: renai
Number of configurations: 6
Output directory: charts\renai

RUN 1/6: temp01
Context: 250 | Temperature: 0.1 | Top-K: None | Top-P: None | Seed: 0
Prediction frames: 25475 (254.8 s)
Window size: 2000 | Context: 250 | Stride: 1750
Window      1-  2000 | Context:    0 | New: 2000 | Active:    0
Window   1751-  3750 | Context:  250 | New: 1750 | Active:    0
Window   3501-  5500 | Context:  250 | New: 1750 | Active:    0
Window   5251-  7250 | Context:  250 | New: 1750 | Active:    0
Window   7001-  9000 | Context:  250 | New: 1750 | Active:    0
Window   8751- 10750 | Context:  250 | New: 1750 | Active:    0
Window  10501- 12500 | Context:  250 | New: 1750 | Active:    0
Window  12251- 14250 | Context:  250 | New: 1750 | Active:    0
Window  14001- 16000 | Context:  250 | New: 1750 | Active:    0
Window  15751- 17750 | Context:  250 | New: 1750 | Active:    0
Window  17501- 19500 | Context:  250 | New: 1750 | Active:    0
Window  19251- 21250 | Context:  250 | 

C:\Users\joshua\vsjycode\csc173\transformers\TRACERS\src\tracers\generate\converter.py:85: UserWarning: Long note end without a start in lane 6 at frame 8362
  warnings.warn(
C:\Users\joshua\vsjycode\csc173\transformers\TRACERS\src\tracers\generate\converter.py:109: UserWarning: Long note in lane 6 starting at beat 509.4800 has no ending
  warnings.warn(


Exported 25475 frame records to: charts\renai\debug_05_temp05_diff5_ctx250_temp0p5_topknone_toppnone_seed0.csv
Saved debug: charts\renai\debug_05_temp05_diff5_ctx250_temp0p5_topknone_toppnone_seed0.csv
Result: 1037/25475 active frames (4.07%)

RUN 6/6: temp06
Context: 250 | Temperature: 0.6 | Top-K: None | Top-P: None | Seed: 0
Prediction frames: 25475 (254.8 s)
Window size: 2000 | Context: 250 | Stride: 1750
Window      1-  2000 | Context:    0 | New: 2000 | Active:   38
Window   1751-  3750 | Context:  250 | New: 1750 | Active:   80
Window   3501-  5500 | Context:  250 | New: 1750 | Active:   82
Window   5251-  7250 | Context:  250 | New: 1750 | Active:   92
Window   7001-  9000 | Context:  250 | New: 1750 | Active:  112
Window   8751- 10750 | Context:  250 | New: 1750 | Active:  108
Window  10501- 12500 | Context:  250 | New: 1750 | Active:   83
Window  12251- 14250 | Context:  250 | New: 1750 | Active:   98
Window  14001- 16000 | Context:  250 | New: 1750 | Active:  116
Window  157

C:\Users\joshua\vsjycode\csc173\transformers\TRACERS\src\tracers\generate\converter.py:72: UserWarning: Unclosed long note in lane 6 before frame 7950
  warnings.warn(
C:\Users\joshua\vsjycode\csc173\transformers\TRACERS\src\tracers\generate\converter.py:85: UserWarning: Long note end without a start in lane 2 at frame 8598
  warnings.warn(
C:\Users\joshua\vsjycode\csc173\transformers\TRACERS\src\tracers\generate\converter.py:85: UserWarning: Long note end without a start in lane 6 at frame 8967
  warnings.warn(
C:\Users\joshua\vsjycode\csc173\transformers\TRACERS\src\tracers\generate\converter.py:72: UserWarning: Unclosed long note in lane 6 before frame 14367
  warnings.warn(
C:\Users\joshua\vsjycode\csc173\transformers\TRACERS\src\tracers\generate\converter.py:72: UserWarning: Unclosed long note in lane 2 before frame 19168
  warnings.warn(
C:\Users\joshua\vsjycode\csc173\transformers\TRACERS\src\tracers\generate\converter.py:85: UserWarning: Long note end without a start in lane 4 

Exported 25475 frame records to: charts\renai\debug_06_temp06_diff5_ctx250_temp0p6_topknone_toppnone_seed0.csv
Saved debug: charts\renai\debug_06_temp06_diff5_ctx250_temp0p6_topknone_toppnone_seed0.csv
Result: 1272/25475 active frames (4.99%)

GENERATION SWEEP SUMMARY
temp01               | Active:     0 | Rate:   0.00%
temp02               | Active:     0 | Rate:   0.00%
temp03               | Active:   319 | Rate:   1.25%
temp04               | Active:     0 | Rate:   0.00%
temp05               | Active:  1037 | Rate:   4.07%
temp06               | Active:  1272 | Rate:   4.99%

All files saved to: charts\renai
